In [1]:
import pandas as pd, numpy as np, glob, joblib, time, os
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

try:
    from xgboost import XGBClassifier
    HAVE_XGB = True
except Exception as e:
    HAVE_XGB = False
print("All imports OK | XGBoost available:", HAVE_XGB)

Matplotlib is building the font cache; this may take a moment.


All imports OK | XGBoost available: False


In [2]:
LABEL_MAP = {
    "BENIGN": "Benign",
    "FTP-Patator": "BruteForce", "SSH-Patator": "BruteForce",
    "DoS Hulk": "DoS", "DoS GoldenEye": "DoS",
    "DoS slowloris": "DoS", "DoS Slowhttptest": "DoS",
    "PortScan": "PortScan",
}

def load_day(path, benign_cap=75_000):
    df = pd.read_csv(path, encoding="latin1")
    df.columns = df.columns.str.strip()            # column names have leading spaces
    df["Label"] = df["Label"].str.strip().map(LABEL_MAP)
    df = df.dropna(subset=["Label"])               # drop labels outside our 4 classes
    feats = df.columns.drop("Label")
    df[feats] = df[feats].replace([np.inf, -np.inf], np.nan).astype("float32")
    df = df.dropna()                               # remove rows with NaN/Inf
    benign = df[df.Label == "Benign"]
    if len(benign) > benign_cap:                   # sample benign down to save RAM
        benign = benign.sample(benign_cap, random_state=42)
    return pd.concat([benign, df[df.Label != "Benign"]])

In [3]:
frames = []
for f in sorted(glob.glob("data/*.csv")):
    t0 = time.time()
    d = load_day(f)
    print(f"{os.path.basename(f):55s} -> {d.shape}  ({time.time()-t0:.0f}s)")
    frames.append(d)

df = pd.concat(frames, ignore_index=True)
before = len(df)
df = df.drop_duplicates()                           # BEFORE splitting, to avoid leakage
print(f"\nDropped {before - len(df):,} duplicate rows")
print(f"Final shape: {df.shape}")
print(df.Label.value_counts())

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv    -> (233804, 79)  (4s)
Monday-WorkingHours.pcap_ISCX.csv                       -> (75000, 79)  (8s)
Tuesday-WorkingHours.pcap_ISCX.csv                      -> (88832, 79)  (7s)
Wednesday-workingHours.pcap_ISCX.csv                    -> (326712, 79)  (11s)

Dropped 141,926 duplicate rows
Final shape: (582422, 79)
Label
Benign        288848
DoS           193730
PortScan       90694
BruteForce      9150
Name: count, dtype: int64


In [ ]:
print("Classes:", sorted(df.Label.unique()))
print("Any NaN left:", df.drop(columns="Label").isna().any().any())
print("Number of features:", df.shape[1] - 1)
print("Memory (MB):", round(df.memory_usage(deep=True).sum() / 1e6))

df.to_pickle("data/clean_2017.pkl")                 # fast reload if the kernel restarts